In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import os
from google.colab import drive #type:ignore
import torch

drive.mount(r'/content/drive/')
df = pd.read_csv(r"/content/drive/MyDrive/DL_CSV/fmnist_small.csv")
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

# Checking for the availability of GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

#Making custom_transform
from torchvision.transforms import transforms

custom_transforms=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225))
])

#Making Custom Dataset class
from PIL import Image
import numpy as np

class CustomDataset(Dataset):

    def __init__(self,features,labels,transforms):
        self.features=features
        self.labels=labels
        self.transforms=transforms

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):

        #resize the image to (28,28)
        image=self.features[index].reshape(28,28)

        #change dataset dtype to uint8
        image=image.astype(np.uint8)

        #convert to 3D image
        image=np.stack([image]*3,axis=-1) #axis=-1 adds the channel parameter at the last

        #Convert to PIL Image
        image=Image.fromarray(image)

        #Apply transformation
        image=self.transforms(image)

        return image,torch.tensor(self.labels[index],dtype=torch.long)
        
# Performing DataSet Split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, shuffle=True)

# Creating CustomDataset Object for train and test
train_dataset = CustomDataset(features=X_train, labels=y_train ,transforms=custom_transforms)
test_dataset = CustomDataset(features=X_test, labels=y_test ,transforms=custom_transforms)

# Creating DataLoader object of the class
# Note: num_workers=2 works fine, but if Colab ever throws a broken pipe error, change it to 0.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,num_workers=0)
   
# Defining Epochs and Learning Rate
epochs = 10
learning_rate = 0.0001

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Using device: cuda


In [11]:
import torchvision.models as models
vgg16=models.vgg16(weights='VGG16_Weights.DEFAULT') 
print(vgg16)
#Freezing the feature extraction Layers
for param in vgg16.features.parameters():
    param.requires_grad=False

#Making the own classifier
vgg16.classifier=nn.Sequential(
    nn.Linear(in_features=25088,out_features=1024),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(in_features=1024,out_features=512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(in_features=512,out_features=10)
)
print(vgg16)
vgg16=vgg16.to(device=device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr=learning_rate)

# Performing Training
for i in range(epochs):
    total_loss = 0

    for features, label in train_loader:
        features, label = features.to(device), label.to(device)
        y_pred = vgg16(features)

        # Making gradient zero
        optimizer.zero_grad()

        # Calculating Loss
        loss = loss_function(y_pred, label)

        # Backward
        loss.backward()

        # Updating Parameters
        optimizer.step()

        total_loss = total_loss + loss.item()
    
    print(f"For epoch :{i+1}, Loss={total_loss/len(train_loader)*100:.2f}")

# Evaluating the vgg16
vgg16.eval() 
total = 0
correct = 0

with torch.no_grad():
    for features, labels in test_loader:
        # FIX: Changed labels(device) to labels.to(device)
        features, labels = features.to(device), labels.to(device)
        y_pred = vgg16(features)

        _, predicted = torch.max(y_pred, 1)
        total = total + labels.shape[0]

        correct = correct + (predicted == labels).sum().item()

print(f"Accuracy of vgg16 = {(correct/total)*100:.2f}%")

For epoch :1, Loss=0.6686915590365727
For epoch :2, Loss=0.2747296163936456
For epoch :3, Loss=0.15722923738261063
For epoch :4, Loss=0.09911562614763776
For epoch :5, Loss=0.05055167941066126
For epoch :6, Loss=0.03834047542729725
For epoch :7, Loss=0.02182051715052997
For epoch :8, Loss=0.013737429524771869
For epoch :9, Loss=0.010163380711649855
For epoch :10, Loss=0.009043449738334553
Accuracy of vgg16 = 89.92%
